# Regression

Fit regression or classification models, use supplied normal equations, and explore polynomial feature maps.

Run the setup cell first. Each numbered section has editable inputs followed by a
calculation cell. Run that section again after changing its inputs, or use
**Restart Kernel and Run All Cells** to run the examples from a clean state.
Run each workflow's calculation before its result-detail cells. Every input has
a comment describing its format or choices. Optional fields accept `None`,
`"None"`, or blank text. Rerun setup after updating these files.

Matrices accept `1 2; 3 4`, rows on separate lines, fractions such as `1/3`,
nested lists, and NumPy arrays. Rows usually represent samples and columns features.
Results shown with `cache=cache` are saved automatically for use in any notebook
through `cache.load("name")`. The [Cache notebook](09_cache.ipynb) manages them.

[Setup and notebook index](README.md)

In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in this folder, the repository root, or a subfolder.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "CommonUtils.py").is_file() and (p / "notebooks" / "notebook_tools.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the ee2211-solver repository or its notebooks folder.")
for folder in (PROJECT_ROOT, PROJECT_ROOT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import numpy as np
import pandas as pd
import sympy as sp
from IPython.display import Markdown, display
import importlib
import notebook_tools
importlib.reload(notebook_tools)
from notebook_tools import (NotebookCache, matrix, optional_matrix, is_blank,
    number, integer, boolean, variable_names, coordinates, choice, show, show_metrics)

pd.set_option("display.precision", 10)
cache = NotebookCache()
print(f"Python {sys.version.split()[0]} | shared cache: {cache.path}")

## 1. Fit from samples
Use short choices: `model="linear"` or `"poly"`, and `target="auto"`, `"numeric"`,
`"sign"`, or `"onehot"`. The original long names are also accepted.
Auto recognizes text labels, ±1 labels, and encoded one-hot rows. Numeric
regression supports multiple output columns.

`bias` controls the constant term for both linear and polynomial models.
`ridge=0` selects ordinary least squares; a positive value selects ridge,
minimizing SSE + ridge × ‖W‖². All coefficients, including the bias, are penalized.
`degree` is used for polynomial models; `form` is only used for positive ridge.

Set `X_test=None` to fit only. Set `Y_test=None` for prediction without test errors.
Blank text and the text `"None"` work too. Invalid supplied matrices and genuinely
unknown test classes still produce a clear error. One-hot training targets may be
labels (numeric or text) or encoded rows. With encoded training rows, test labels
can be zero-based class column indices.

In [ ]:
X = '1 3 -2; -4 0 -1; 3 1 8; 2 1 6; 8 4 6'  # Samples by feature; spaces separate columns, ; separates rows.
Y = '1;1;2;3;3'  # One numeric/text label per row, or one-hot/numeric output rows.
X_test = '1 -2 4'  # Prediction rows with the same features, or None to skip.
Y_test = None  # Optional true targets: None, "None", or "" skips test errors.
model = 'linear'  # "linear" | "poly".
target = 'onehot'  # "auto" | "numeric" | "sign" | "onehot".
bias = False  # True adds a constant; False excludes it (linear and polynomial).
ridge = 0  # 0 = ordinary least squares; positive number = ridge penalty.
degree = 2  # Polynomial order, integer >= 1; ignored for linear models.
form = 'auto'  # "auto" | "primal" | "dual"; used only when ridge > 0.

In [ ]:
from notebook_tools import fit_model, fit_regression_model

regression = fit_model(X, Y, X_test, Y_test, model=model, target=target,
                       bias=bias, ridge=ridge, degree=degree, form=form)
show_metrics({key: regression[key] for key in ("model_label", "target_mode", "system", "form", "class_labels")})
for key in ("model_matrix", "target_matrix", "w", "train_pred", "train_class", "sse", "mse", "test_pred", "test_class", "test_target_matrix", "test_sse", "test_mse"):
    show("regression_" + key, regression[key], cache=cache, source="Fit from samples")

### Normal equations and error checks
SSE and MSE above are per output. Total SSE sums over all samples and outputs.
The objective gradient should be close to zero at an optimum. For ordinary least
squares XᵀE is close to zero; ridge stationarity also includes λW.

In [ ]:
diagnostics = regression["diagnostics"]
show_metrics({key: diagnostics[key] for key in ("total_sse", "sse_per_sample", "objective")})
for key in ("gram", "rhs", "residual", "normal_residual", "objective_gradient"):
    show(key, diagnostics[key])
show_metrics(diagnostics["shapes"])

### Worked variants
These independent recipes exercise polynomial, ridge, ridge-polynomial,
one-hot, and sign classification. One-hot predictions use the largest score;
ties choose the first class column. A sign score of zero gives label zero.
Edit a recipe's arguments to try another example.

In [ ]:
variant_inputs = {
    "polynomial": dict(X=[[0], [1], [2]], Y=[[1], [4], [9]], X_test=[[3]], model_type="Polynomial", target_mode="Numeric regression"),
    "ridge": dict(X=[[0], [1]], Y=[[1], [3]], regularization="Ridge", target_mode="Numeric regression"),
    "ridge_polynomial": dict(X=[[0], [1], [2]], Y=[[1], [4], [9]], X_test=[[3]], model_type="Polynomial", regularization="Ridge", target_mode="Numeric regression"),
    "one_hot": dict(X=[[0], [1], [2], [3]], Y=[["low"], ["low"], ["high"], ["high"]], X_test=[[2.5]], Y_test=[["high"]], target_mode="One-hot classification"),
    "sign": dict(X=[[-1], [1]], Y=[[-1], [1]], X_test=[[-2], [0], [2]], include_bias=False, target_mode="Sign classification"),
}
variants = {}
for name, arguments in variant_inputs.items():
    fitted = fit_regression_model(**arguments)
    variants[name] = fitted
    display(Markdown("#### " + name.replace("_", " ").title()))
    show("weights", fitted["w"])
    show("prediction", fitted["test_pred"] if fitted["test_pred"] is not None else fitted["train_pred"])
    show("predicted_class", fitted["test_class"])

## 2. Use normal equation summaries
Enter supplied XᵀX and XᵀY directly. Use λ = 0 for ordinary least squares.
Prediction rows must already contain constant or polynomial terms in the same
column order as the summaries. Training SSE cannot be recovered from these two
summary matrices alone. Set `prediction_rows_raw = None` to skip prediction.

In [ ]:
G = '4 0; 0 9'  # Supplied X.T @ X, a symmetric square matrix.
H = '8; 18'  # Supplied X.T @ Y; one row per model coefficient.
penalty = 1.0  # Nonnegative ridge penalty; 0 selects OLS.
rows = '1 1'  # Optional design rows for prediction; None skips.

In [ ]:
from RegressionUtils import solve_normal_equations

normal_equations = solve_normal_equations(matrix(G), matrix(H), number(penalty))
for key in ("w", "regularized_gram", "normal_residual"):
    show("summary_" + key, normal_equations[key], cache=cache, source="Normal equation summaries")
prediction_rows = optional_matrix(rows)
if prediction_rows is not None:
    if prediction_rows.shape[1] != normal_equations["w"].shape[0]:
        raise ValueError("Prediction rows need one column per coefficient.")
    summary_prediction = prediction_rows @ normal_equations["w"]
    show("summary_prediction", summary_prediction, cache=cache, source="Normal equation prediction")

## 3. Explore polynomial features
Count monomials, show their column order, optionally transform sample points,
and inspect primal/dual system dimensions without fitting a model.
Set `feature_points_raw = None` for counts and names only.

In [ ]:
d = 3  # Number of raw features, integer >= 1.
power = 2  # Maximum polynomial order, integer >= 1.
constant = True  # True includes the constant monomial; False excludes it.
points = '2 -1 3'  # Rows with d features, or None for names/counts only.
n = 9  # Positive training-sample count; for dimension estimates.
outputs = 1  # Positive target-column count; for dimension estimates.

In [ ]:
from RegressionUtils import polynomial_features_details, regression_shapes

features = polynomial_features_details(integer(d), integer(power),
    optional_matrix(points), boolean(constant))
print(f"Monomial coefficients: {features['count']}")
display(pd.DataFrame({"Column (1-based)": np.arange(1, features["count"] + 1), "Monomial": features["names"]}))
show("polynomial_features", features["transformed"], cache=cache, source="Polynomial feature map")
show_metrics(regression_shapes(integer(n), features["count"], integer(outputs)))